# fabric-ai-meta Quickstart

This notebook walks through the core capabilities of `fabric-ai-meta` inside a Microsoft Fabric notebook.

**Prerequisites:**
- A Microsoft Fabric workspace with one or more semantic models
- This notebook running inside a Fabric notebook environment

**What you will do:**
1. Install the package
2. Authenticate (automatic in Fabric)
3. List available semantic models
4. Analyze and score a model
5. Generate an AI-ready schema
6. Export for AI frameworks (LangChain, OpenAI)
7. Run a cross-model governance report

## 1. Install

Install `fabric-ai-meta` into this notebook session.

In [ ]:
%pip install fabric-ai-meta

## 2. Authenticate

Inside Fabric, authentication is automatic via the ambient notebook credential. This cell verifies you are running in the correct environment.

In [ ]:
from fabric_ai_meta.auth.entra import detect_notebook_environment, get_credential

assert detect_notebook_environment(), "This notebook must run inside Microsoft Fabric"
credential = get_credential()
print("Authenticated successfully")

## 3. List Models

Discover all semantic models in your workspace. Replace `YOUR_WORKSPACE_NAME` with your actual workspace name.

In [ ]:
from fabric_ai_meta.extractor.semantic_link import SemanticLinkExtractor

WORKSPACE = "YOUR_WORKSPACE_NAME"  # <-- Replace with your workspace name

extractor = SemanticLinkExtractor()
models = extractor.list_models(WORKSPACE)
print(f"Found {len(models)} models: {models}")

## 4. Analyze a Single Model

Extract metadata, classify tables and measures, then compute an AI readiness score. The score (0.0 to 1.0) reflects how well the model is prepared for AI consumption: description coverage, relationship completeness, measure documentation, and more.

In [ ]:
from fabric_ai_meta import score_model
from fabric_ai_meta.analyzer.pipeline import classify_model_in_place

model = extractor.extract(models[0], WORKSPACE)

# classify_model_in_place is the same path the CLI uses: it assigns table types,
# column roles, and measure categories, AND populates measure dependencies
# (depends_on_measures / depends_on_columns / implicit_filters). Classifying by
# hand instead drops those, and the AI-ready schema loses its `depends_on` block.
classify_model_in_place(model)

score, breakdown = score_model(model)
model.ai_readiness_score = score
model.scoring_breakdown = breakdown

print(f"Model: {model.name}")
print(f"AI Readiness Score: {score:.2f}")
print(f"Tables: {len(model.tables)}, Measures: {sum(len(t.measures) for t in model.tables)}")
print(f"Breakdown: {breakdown}")

## 5. Generate AI-Ready Schema

The AI-ready schema is a structured JSON file that describes the model's tables, columns, measures, relationships, and query guidance. It is designed for consumption by LLM agents and AI frameworks.

In [ ]:
from fabric_ai_meta import generate_ai_ready_schema
from fabric_ai_meta.generator.schema import write_schema_to_file

schema = generate_ai_ready_schema(model)

slug = model.name.lower().replace(" ", "-")
write_schema_to_file(model, f"{slug}/ai-ready-schema.json")

print(f"Schema written with {len(schema['tables'])} tables and {len(schema['measures'])} measures")
print(f"Scoring: {schema['scoring']['overall']:.2f}")
print(f"Pitfalls: {len(schema['query_guidance']['common_pitfalls'])}")

## 6. Export for AI Frameworks

Generate tool definitions for LangChain and OpenAI function calling. These JSON structures can be passed directly to AI agents so they understand how to query this semantic model.

In [ ]:
from fabric_ai_meta import to_langchain_tool_definition, to_openai_function

langchain_def = to_langchain_tool_definition(model)
openai_def = to_openai_function(model)

print("=== LangChain Tool ===")
print(f"Name: {langchain_def['name']}")
print(f"Description: {langchain_def['description'][:100]}...")

print("\n=== OpenAI Function ===")
print(f"Name: {openai_def['function']['name']}")
print(f"Required params: {openai_def['function']['parameters']['required']}")

## 7. Cross-Model Governance Report

Analyze all models in the workspace for naming inconsistencies, duplicate measures, and overall readiness. This is most valuable when you have multiple semantic models in a workspace.

In [ ]:
from fabric_ai_meta import generate_governance_report

all_models = []
for name in models:
    m = extractor.extract(name, WORKSPACE)
    classify_model_in_place(m)
    s, b = score_model(m)
    m.ai_readiness_score = s
    m.scoring_breakdown = b
    all_models.append(m)

report = generate_governance_report(all_models)

print(f"Models analyzed: {report['summary']['model_count']}")
print(f"Naming issues: {report['summary']['total_naming_issues']}")
print(f"Duplicate measures: {report['summary']['total_duplicate_measures']}")
print("\nRecommendations:")
for rec in report['recommendations']:
    print(f"  - {rec}")

## 8. Do You Even Need an Ontology?

Knowledge graph projects are expensive and most semantic models do not need one. `assess_graph_necessity` scores whether a model's workload actually justifies one, using metadata you have already extracted. No extra Fabric capacity, no LLM calls.

Pass the questions your users really ask; without them the check falls back to Copilot example prompts, then to measure dependencies, and reports lower confidence.

In [ ]:
from fabric_ai_meta import assess_graph_necessity

QUESTIONS = [
    # Replace with real questions your users ask, phrased with table/column names
    # that exist in the model. Fewer than half matching drops confidence to
    # "directional".
]

verdict = assess_graph_necessity(model, questions=QUESTIONS or None)

print(f"{verdict['name']}: {verdict['tier']}  "
      f"(pressure {verdict['pressure']:.2f}, confidence {verdict['confidence']})")
for line in verdict["evidence"]:
    print(f"  - {line}")
print(f"\n{verdict['recommendation']}")

## 9. Run It Without Fabric

Everything above needs the Fabric runtime because live workspace extraction does. The rest of the tool does not: point it at a Power BI project folder on any machine and you get the same outputs with no tenant and no sign-in.

In Power BI Desktop: **File > Save As > Power BI project (.pbip)**, then from a terminal:

```bash
fabric-ai-meta analyze "Your Model" --pbip ./YourModel.SemanticModel
fabric-ai-meta governance --pbip ./git-integration-repo --graph-necessity --report ./gov.json
```

Or from Python, swapping the extractor and keeping every other line the same:

```python
from fabric_ai_meta import PbipExtractor
model = PbipExtractor("./YourModel.SemanticModel").extract("Your Model")
```

## Next Steps

- **CLI usage:** Run `fabric-ai-meta analyze`, `scan`, `export`, `governance` from the terminal
- **LLM enrichment:** Add `--llm-enrich` to auto-generate missing descriptions. Any of 10+ providers (Anthropic, OpenAI, Google, Bedrock, Azure, local Ollama) via `.fabric-ai-meta.toml`
- **Prep for AI:** Run `fabric-ai-meta export prep-for-ai` to generate settings for the Fabric UI
- **Additional frameworks:** Export for Semantic Kernel (`to_semantic_kernel_plugin`) and AutoGen (`to_autogen_tool`)
- **Prep for AI writeback:** `export copilot` snapshots the live `Copilot/` folder, `apply-copilot` writes an edited one back
- **Full documentation:** The [user guide](https://github.com/psistla/fabric-ai-meta/blob/master/docs/user-guide.md) is the per-command reference